In [1]:
import pandas as pd
import requests
from io import StringIO

try:
    r = requests.get(r'https://bit.ly/fish_csv_data', timeout=10)
    r.raise_for_status()
    fish = pd.read_csv(StringIO(r.text), low_memory=False)
except Exception as e:
    print(e)

if fish is not None:
    print(fish.head())
    print(fish.describe())
    print(fish.info())
    print(fish.columns)
    print(fish['Species'].unique())
    
fish_input = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']]
fish_target = fish['Species']

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV, cross_validate
train_input, test_input, train_target, test_target = train_test_split(fish_input, fish_target, stratify=fish_target, random_state=42)
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

from sklearn.neighbors import KNeighborsClassifier
import numpy as np
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kn = KNeighborsClassifier(n_neighbors=3)
scores = cross_validate(kn, train_scaled, train_target, cv=cv, n_jobs=1, return_train_score=True)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

kn.fit(train_scaled, train_target)
print(kn.score(train_scaled, train_target))
print(kn.score(test_scaled, test_target))
print(kn.predict(test_scaled[:5]))
print(kn.classes_)
print(kn.predict_proba(test_scaled[:5]))

print(np.round(kn.predict_proba(test_scaled[4:5]), decimals=3))
print(kn.classes_)
distances, idxs = kn.kneighbors(test_scaled[4:5])
print(idxs[0])
print(train_target.iloc[idxs[0]])


  Species  Weight  Length  Diagonal   Height   Width
0   Bream   242.0    25.4      30.0  11.5200  4.0200
1   Bream   290.0    26.3      31.2  12.4800  4.3056
2   Bream   340.0    26.5      31.1  12.3778  4.6961
3   Bream   363.0    29.0      33.5  12.7300  4.4555
4   Bream   430.0    29.0      34.0  12.4440  5.1340
            Weight      Length    Diagonal      Height       Width
count   159.000000  159.000000  159.000000  159.000000  159.000000
mean    398.326415   28.415723   31.227044    8.970994    4.417486
std     357.978317   10.716328   11.610246    4.286208    1.685804
min       0.000000    8.400000    8.800000    1.728400    1.047600
25%     120.000000   21.000000   23.150000    5.944800    3.385650
50%     273.000000   27.300000   29.400000    7.786000    4.248500
75%     650.000000   35.500000   39.650000   12.365900    5.584500
max    1650.000000   63.400000   68.000000   18.957000    8.142000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data co

In [14]:
train_target.value_counts()
idx = (train_target == 'Perch') | (train_target == 'Bream')
train_pb = train_scaled[idx]
target_pb = train_target[idx]

idx2 = (test_target == 'Perch') | (test_target == 'Bream')
test_pb = test_scaled[idx2]
test_target_pb = test_target[idx2]

from sklearn.linear_model import LogisticRegression
from scipy.special import expit
lr = LogisticRegression(C=20, solver='lbfgs', penalty='l2', max_iter=1000)
lr.fit(train_pb, target_pb)
try:    
    print(lr.classes_)
    print(lr.predict(test_pb[:5]))
    print(np.round(lr.predict_proba(test_pb[:5]), decimals=3))
    print(lr.coef_, lr.intercept_)
except Exception as e:
    print(e)

['Bream' 'Perch']
['Perch' 'Perch' 'Perch' 'Perch' 'Bream']
[[0.015 0.985]
 [0.012 0.988]
 [0.027 0.973]
 [0.005 0.995]
 [0.956 0.044]]
[[ 1.3732418   1.67998521 -0.56515732 -7.08474131  2.04951226]] [2.73873239]


C:\Users\박중현\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [23]:
mlr = LogisticRegression(C=60, solver='newton-cholesky', max_iter=1000, penalty='l2')
mlr.fit(train_scaled, train_target)
print(mlr.coef_, mlr.intercept_)
print(mlr.classes_)
print(mlr.predict(test_scaled[:5]))
print(np.round(mlr.predict_proba(test_scaled[:5]), decimals=3))
print(mlr.decision_function(test_scaled[:5]))

[[ -2.95736445  -2.63420896   4.52210428  11.47363534  -0.9276628 ]
 [  0.9072502   -1.55462004  -5.93873158   9.00148896  -2.3177243 ]
 [  3.89180137  12.31699611 -14.63986502  -8.24855287   5.48047409]
 [ -0.04374643   4.620567     4.73554846  -4.63571602  -2.99459971]
 [ -2.13465427 -10.58250357   9.18224268  -1.39311529   3.24042075]
 [ -2.87936844   2.83323743   2.26136229  -8.21011058  -5.13489247]
 [  3.21608203  -4.99946796  -0.12266112   2.01237046   2.65398442]] [-0.34009488  0.39445957  2.96099211 -0.39379816  2.6973719  -8.87297183
  3.55404129]
['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']
['Roach' 'Perch' 'Perch' 'Parkki' 'Parkki']
[[0.    0.012 0.117 0.001 0.833 0.004 0.033]
 [0.    0.019 0.628 0.    0.311 0.001 0.041]
 [0.    0.04  0.647 0.    0.278 0.007 0.027]
 [0.    0.973 0.    0.    0.015 0.    0.011]
 [0.    0.945 0.    0.    0.04  0.    0.015]]
[[-6.50568295  0.36766951  2.6371859  -1.79185085  4.6014798  -0.67439568
   1.36559428]
 [-6.43750894  

C:\Users\박중현\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [69]:
from sklearn.linear_model import SGDClassifier
sgd = SGDClassifier(loss='modified_huber', max_iter=300, tol=None, random_state=42)
sgd.fit(train_scaled, train_target)
print(sgd.score(train_scaled, train_target))
print(sgd.score(test_scaled, test_target))

from sklearn.preprocessing import PolynomialFeatures
p = PolynomialFeatures()

0.9663865546218487
0.95
